# Práctica N° 6 — Búsqueda Semántica con ChromaDB y LangChain

Se indexa el corpus RENACE en ChromaDB a través de LangChain y se mide Precision@5 y latencia sobre el mismo `eval_set` de 10 queries, insumo para la comparación del Ejercicio 4/5 de la Práctica 9.

In [1]:
import os, warnings

os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
warnings.filterwarnings('ignore')

!pip install langchain-chroma langchain-huggingface sentence-transformers pydantic-settings --quiet --disable-pip-version-check

import time
import numpy as np
import pandas as pd
import chromadb
from huggingface_hub.utils import logging as hf_logging
hf_logging.set_verbosity_error()
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

## Corpus e indexación

In [ ]:
corpus = [
    # EDA (10 documentos)
    {'titulo':'EDA en menores: rehidratación oral y zinc',
     'resumen':'La enfermedad diarreica aguda en menores de 5 años se trata con sales de rehidratación oral y zinc oral por 10-14 días. La lactancia materna exclusiva reduce el riesgo en un 50%.',
     'categoria':'EDA','año':2024,'region':'Nacional','fuente':'RENACE-MINSA'},
    {'titulo':'Rotavirus: principal agente de EDA hospitalaria',
     'resumen':'Rotavirus causa el 40% de hospitalizaciones por diarrea en Perú. La vacuna redujo la mortalidad pediátrica por EDA en 35% desde 2009. Cobertura vacunal actual: 87%.',
     'categoria':'EDA','año':2024,'region':'Lima','fuente':'RENACE-MINSA'},
    {'titulo':'Brote EDA asociado a agua contaminada Cajamarca',
     'resumen':'Brote de 234 casos en provincia de Cajamarca vinculado a sistema de agua sin cloración. Agente: E. coli enterotoxigénica. Tasa de ataque: 18% de la población expuesta.',
     'categoria':'EDA','año':2023,'region':'Cajamarca','fuente':'RENACE-MINSA'},
    {'titulo':'Canal endémico EDA semanas epidemiológicas Arequipa',
     'resumen':'El análisis del canal endémico de EDA en Arequipa 2019-2024 muestra zona de epidemia en semanas 3-8 (verano). Correlación con precipitaciones pluviales r=0.71.',
     'categoria':'EDA','año':2024,'region':'Arequipa','fuente':'RENACE-MINSA'},
    {'titulo':'ETEC y cólera: diagnóstico diferencial en zona endémica',
     'resumen':'La E. coli enterotoxigénica produce diarrea acuosa indistinguible del cólera clínicamente. El cultivo diferencial y PCR son necesarios para el diagnóstico etiológico.',
     'categoria':'EDA','año':2023,'region':'Loreto','fuente':'DGE-MINSA'},
    {'titulo':'EDA en comunidades indígenas amazónicas: barreras de acceso al agua potable',
     'resumen':'El 62% de comunidades indígenas de Loreto y Ucayali carecen de acceso a agua potable segura, lo que triplica la incidencia de EDA respecto al promedio nacional. Programas de cloración domiciliaria redujeron casos en 28% en pilotos 2023-2024.',
     'categoria':'EDA','año':2024,'region':'Ucayali','fuente':'RENACE-MINSA'},
    {'titulo':'Saneamiento básico rural y su impacto en la incidencia de EDA',
     'resumen':'Distritos rurales sin conexión a alcantarillado presentan una incidencia de EDA 2.4 veces mayor que distritos con saneamiento completo. La inversión en letrinas mejoradas se asocia a una reducción del 31% en casos infantiles.',
     'categoria':'EDA','año':2023,'region':'Cusco','fuente':'RENACE-MINSA'},
    {'titulo':'Vigilancia de cólera en Perú: dos décadas sin casos autóctonos',
     'resumen':'Perú no reporta casos autóctonos de cólera desde 2003, pero mantiene vigilancia activa por el riesgo de reintroducción desde países vecinos. El sistema RENACE incluye cólera como enfermedad de notificación obligatoria inmediata.',
     'categoria':'EDA','año':2024,'region':'Nacional','fuente':'DGE-MINSA'},
    {'titulo':'Cambio climático y proyección de brotes de EDA en la costa peruana',
     'resumen':'Modelos climáticos proyectan un incremento del 22% en brotes de EDA costeros para 2030 asociado a eventos El Niño más frecuentes e intensos. Las regiones de Piura y Tumbes son las de mayor vulnerabilidad proyectada.',
     'categoria':'EDA','año':2024,'region':'Piura','fuente':'INS-MINSA'},
    {'titulo':'Suplementación con probióticos como coadyuvante en el tratamiento de EDA infantil',
     'resumen':'Un ensayo clínico en Lima mostró que Lactobacillus rhamnosus GG reduce la duración de la diarrea aguda en 1.2 días en promedio cuando se administra junto con SRO. No reemplaza la rehidratación oral estándar.',
     'categoria':'EDA','año':2023,'region':'Lima','fuente':'INS-MINSA'},

    # Dengue (10 documentos)
    {'titulo':'Brote dengue Loreto: 3500 casos serotipo DENV-2',
     'resumen':'Loreto registra 3500 casos de dengue en la primera mitad de 2024, predominando DENV-2. Índice aédico del 8% en zona urbana de Iquitos. 12 fallecidos por dengue grave.',
     'categoria':'Dengue','año':2024,'region':'Loreto','fuente':'DGE-MINSA'},
    {'titulo':'Coinfección dengue-COVID en región Piura',
     'resumen':'Se documentaron 89 casos de coinfección dengue-COVID-19 en Piura 2021. La coinfección aumentó el riesgo de dengue grave (OR=2.3). PCR multiplex permite diagnóstico en 4 horas.',
     'categoria':'Dengue','año':2023,'region':'Piura','fuente':'DGE-MINSA'},
    {'titulo':'Control vectorial Aedes aegypti campaña 2024',
     'resumen':'La campaña nacional de eliminación de criaderos redujo el índice aédico del 6.2% al 2.8% en las 12 ciudades intervenidas. Fumigación adulticida complementaria aplicada en 98,000 viviendas.',
     'categoria':'Dengue','año':2024,'region':'Nacional','fuente':'DGE-MINSA'},
    {'titulo':'Vacuna dengue Dengvaxia: programa piloto Iquitos',
     'resumen':'El programa piloto de vacunación con Dengvaxia en Iquitos 2023 alcanzó cobertura del 72% en población seropositiva de 9-45 años. Reducción de hospitalizaciones del 80%.',
     'categoria':'Dengue','año':2024,'region':'Loreto','fuente':'DGE-MINSA'},
    {'titulo':'Modelado matemático dengue con temperatura El Niño',
     'resumen':'Modelo SIR ampliado con temperatura predice brotes de dengue con 6 semanas de anticipación. El Niño aumenta la capacidad vectorial del Aedes en 340% en costa norte peruana.',
     'categoria':'Dengue','año':2024,'region':'Nacional','fuente':'INS-MINSA'},
    {'titulo':'Dengue grave: signos de alarma y protocolo de manejo hospitalario',
     'resumen':'Los signos de alarma del dengue incluyen dolor abdominal intenso, vómitos persistentes, sangrado de mucosas y letargo. Su identificación temprana reduce la mortalidad por dengue grave de 5% a menos de 1% con manejo hospitalario oportuno.',
     'categoria':'Dengue','año':2024,'region':'Lima','fuente':'DGE-MINSA'},
    {'titulo':'Circulación simultánea de cuatro serotipos de dengue en la costa norte',
     'resumen':'Piura y Tumbes reportan circulación simultánea de DENV-1, DENV-2, DENV-3 y DENV-4 en 2024, incrementando el riesgo de dengue grave por infecciones secundarias heterotípicas. La vigilancia serotípica se realiza mediante RT-PCR.',
     'categoria':'Dengue','año':2024,'region':'Piura','fuente':'DGE-MINSA'},
    {'titulo':'Resistencia a insecticidas piretroides en poblaciones de Aedes aegypti',
     'resumen':'Estudios entomológicos en Iquitos confirman resistencia a deltametrina en el 68% de poblaciones de Aedes aegypti evaluadas. Se recomienda rotación con organofosforados y refuerzo del control mecánico de criaderos.',
     'categoria':'Dengue','año':2023,'region':'Loreto','fuente':'INS-MINSA'},
    {'titulo':'Dengue en gestantes: riesgo de complicaciones perinatales',
     'resumen':'El dengue durante el embarazo se asocia a mayor riesgo de parto prematuro y bajo peso al nacer (OR=1.8). La transmisión vertical es infrecuente pero posible cuando la infección ocurre cerca del parto.',
     'categoria':'Dengue','año':2023,'region':'Loreto','fuente':'INS-MINSA'},
    {'titulo':'Sistema de vigilancia entomológica: índices larvarios y aédico en 24 ciudades',
     'resumen':'El monitoreo entomológico continuo en 24 ciudades prioritarias mostró índice aédico promedio de 4.1% en 2024. Ciudades con vigilancia comunitaria activa mantienen índices bajo 2%, el umbral de riesgo epidémico.',
     'categoria':'Dengue','año':2024,'region':'Nacional','fuente':'DGE-MINSA'},

    # Tuberculosis (10 documentos)
    {'titulo':'TB-MDR Lima: 1200 nuevos casos en 2024',
     'resumen':'Lima concentra 1200 de los 1890 casos de TB-MDR registrados en Perú en 2024. Resistencia a isoniazida y rifampicina en el 8.5% de casos nuevos. GeneXpert detecta TB-MDR en 2 horas.',
     'categoria':'Tuberculosis','año':2024,'region':'Lima','fuente':'PCTB-MINSA'},
    {'titulo':'Tratamiento TB-XDR con bedaquilina y pretomanid',
     'resumen':'El esquema BPaLM (bedaquilina, pretomanid, linezolid, moxifloxacino) muestra 89% de éxito en TB-XDR en ensayo multicéntrico peruano. Duración: 6 meses vs 24 meses del esquema clásico.',
     'categoria':'Tuberculosis','año':2024,'region':'Lima','fuente':'INS-MINSA'},
    {'titulo':'Coinfección TB-VIH: protocolo de tratamiento simultáneo',
     'resumen':'La coinfección TB-VIH representa el 7% de casos en Perú. El inicio simultáneo de TARGA y tratamiento TB reduce mortalidad en 56%. IRIS es la complicación más frecuente (38%).',
     'categoria':'Tuberculosis','año':2023,'region':'Nacional','fuente':'PCTB-MINSA'},
    {'titulo':'GeneXpert Ultra: nueva prueba diagnóstica TB infantil',
     'resumen':'GeneXpert MTB/RIF Ultra detecta TB en muestra extrapulmonar infantil con sensibilidad del 84% vs 52% de la baciloscopia. Permite diagnóstico pediátrico en zonas rurales sin cultivo.',
     'categoria':'Tuberculosis','año':2024,'region':'Nacional','fuente':'INS-MINSA'},
    {'titulo':'DOTS y adherencia TB: análisis San Juan de Lurigancho',
     'resumen':'Estudio en SJL muestra 91% de éxito terapéutico con DOTS supervisado vs 74% con auto-administración. Abandono asociado a distancia al establecimiento > 30 minutos.',
     'categoria':'Tuberculosis','año':2023,'region':'Lima','fuente':'PCTB-MINSA'},
    {'titulo':'Tratamiento preventivo de tuberculosis latente con isoniazida',
     'resumen':'Se estima que el 25% de la población peruana tiene infección tuberculosa latente. El tratamiento preventivo con isoniazida durante 6 meses reduce el riesgo de progresión a enfermedad activa en 60-90%, priorizando contactos y personas con VIH.',
     'categoria':'Tuberculosis','año':2024,'region':'Nacional','fuente':'PCTB-MINSA'},
    {'titulo':'Tuberculosis en establecimientos penitenciarios: hacinamiento y transmisión',
     'resumen':'La incidencia de TB en penales peruanos es 15 veces mayor que en población general, atribuida a hacinamiento y ventilación deficiente. Programas de tamizaje activo bianual redujeron el subdiagnóstico en 40%.',
     'categoria':'Tuberculosis','año':2023,'region':'Lima','fuente':'PCTB-MINSA'},
    {'titulo':'Inteligencia artificial aplicada al tamizaje radiológico de tuberculosis',
     'resumen':'Un algoritmo de deep learning para lectura de radiografías de tórax alcanzó sensibilidad de 94% para detectar hallazgos sugestivos de TB pulmonar, apoyando el tamizaje masivo en zonas con escasez de radiólogos.',
     'categoria':'Tuberculosis','año':2024,'region':'Nacional','fuente':'INS-MINSA'},
    {'titulo':'Estado nutricional y desenlace del tratamiento antituberculoso',
     'resumen':'Pacientes con desnutrición al inicio del tratamiento TB tienen 2.1 veces más riesgo de abandono o fracaso terapéutico. La suplementación nutricional durante la fase intensiva mejora la adherencia y la ganancia de peso.',
     'categoria':'Tuberculosis','año':2023,'region':'Arequipa','fuente':'PCTB-MINSA'},
    {'titulo':'Cobertura de vacunación BCG neonatal y protección contra formas graves de TB',
     'resumen':'La cobertura de BCG en recién nacidos supera el 90% a nivel nacional. La vacuna protege principalmente contra formas graves de TB infantil (meningitis TB, TB miliar), con eficacia limitada contra TB pulmonar en adultos.',
     'categoria':'Tuberculosis','año':2024,'region':'Nacional','fuente':'INS-MINSA'},

    # Malaria (10 documentos)
    {'titulo':'Malaria Plasmodium vivax: reemergencia Piura 2024',
     'resumen':'Piura registra 2,340 casos de P. vivax en 2024, el mayor número desde 2017. El 73% de casos ocurren en zonas rurales con escasa cobertura del programa de prevención vectorial.',
     'categoria':'Malaria','año':2024,'region':'Piura','fuente':'DGE-MINSA'},
    {'titulo':'Resistencia cloroquina P. falciparum en Loreto',
     'resumen':'El 23% de casos de P. falciparum en Loreto presenta resistencia a cloroquina. Primera línea actualizada a arteméter-lumefantrina. Vigilancia molecular con microsatélites confirma propagación clonal.',
     'categoria':'Malaria','año':2024,'region':'Loreto','fuente':'INS-MINSA'},
    {'titulo':'Malaria gestacional: impacto en bajo peso al nacer',
     'resumen':'La malaria en gestantes aumenta el riesgo de bajo peso al nacer en 2.8 veces (RR=2.8, IC 95%: 1.9-4.1). Tamizaje con gota gruesa en cada control prenatal reduce complicaciones en 65%.',
     'categoria':'Malaria','año':2023,'region':'Loreto','fuente':'INS-MINSA'},
    {'titulo':'Tafenoquina para eliminación de hipnozoítos P. vivax',
     'resumen':'Tafenoquina monodosis (300 mg) es tan eficaz como primaquina 14 días para la curación radical de P. vivax. Mayor adherencia y menor riesgo de recaídas documentado en ensayo peruano.',
     'categoria':'Malaria','año':2024,'region':'Nacional','fuente':'INS-MINSA'},
    {'titulo':'Sistema de alerta temprana malaria clima Amazonia',
     'resumen':'Modelo predictivo basado en NDVI satelital, temperatura y precipitación anticipa brotes de malaria en Loreto con 8 semanas de anticipación. Sensibilidad del 88%, especificidad del 79%.',
     'categoria':'Malaria','año':2024,'region':'Loreto','fuente':'INS-MINSA'},
    {'titulo':'Comparación de microscopía y pruebas rápidas para diagnóstico de malaria',
     'resumen':'Las pruebas rápidas (RDT) para malaria muestran sensibilidad de 95% para P. falciparum pero menor para P. vivax (82%). La gota gruesa sigue siendo el estándar de oro para cuantificar parasitemia y confirmar especie.',
     'categoria':'Malaria','año':2024,'region':'Loreto','fuente':'INS-MINSA'},
    {'titulo':'Reemergencia de malaria urbana en zonas periurbanas de Iquitos',
     'resumen':'Se documentó transmisión de malaria en zonas periurbanas de Iquitos previamente consideradas de bajo riesgo, asociada a criaderos en pozas de agua estancada por deficiente drenaje pluvial urbano.',
     'categoria':'Malaria','año':2023,'region':'Loreto','fuente':'DGE-MINSA'},
    {'titulo':'Resistencia a insecticidas en Anopheles darlingi de la Amazonía peruana',
     'resumen':'Poblaciones de Anopheles darlingi en Loreto muestran resistencia emergente a piretroides, comprometiendo la efectividad de mosquiteros tratados con insecticida (LLINs). Se recomienda evaluación periódica de susceptibilidad.',
     'categoria':'Malaria','año':2024,'region':'Loreto','fuente':'INS-MINSA'},
    {'titulo':'Malaria importada en viajeros: vigilancia epidemiológica en puntos de entrada',
     'resumen':'Los casos de malaria importada representan el 3% del total nacional, principalmente en viajeros retornantes de la Amazonía hacia Lima. La vigilancia en aeropuertos permite diagnóstico y tratamiento oportuno evitando transmisión secundaria.',
     'categoria':'Malaria','año':2023,'region':'Lima','fuente':'DGE-MINSA'},
    {'titulo':'Impacto económico de la malaria en comunidades mineras informales de Madre de Dios',
     'resumen':'La malaria genera pérdidas económicas estimadas en 3.2 millones de soles anuales en comunidades mineras informales de Madre de Dios, por ausentismo laboral y gastos de tratamiento. La deforestación asociada a minería incrementa criaderos del vector.',
     'categoria':'Malaria','año':2024,'region':'Madre de Dios','fuente':'INS-MINSA'},
]

documentos, metadatos, ids = [], [], []
for i, doc in enumerate(corpus):
    documentos.append(f"{doc['titulo']}. {doc['resumen']}")
    metadatos.append({
        'categoria': doc['categoria'], 'año': doc['año'],
        'region': doc['region'], 'fuente': doc['fuente'],
    })
    ids.append(f"{doc['categoria'].lower()}_{i+1:02d}")

print(f'Corpus: {len(documentos)} documentos')
print(pd.Series([m['categoria'] for m in metadatos]).value_counts())

Corpus: 40 documentos
EDA             10
Dengue          10
Tuberculosis    10
Malaria         10
Name: count, dtype: int64


In [3]:
embedding_fn = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

client = chromadb.EphemeralClient(settings=chromadb.Settings(anonymized_telemetry=False))
vectorstore = Chroma(client=client, collection_name='renace_lab06', embedding_function=embedding_fn)
vectorstore.add_texts(texts=documentos, metadatas=metadatos, ids=ids)
print(f'Documentos indexados: {vectorstore._collection.count()}')

Documentos indexados: 40


## Búsqueda semántica

In [4]:
retriever = vectorstore.as_retriever(search_type='similarity', search_kwargs={'k': 5})

query = '¿Cómo prevenir la diarrea en niños pequeños?'
resultados = retriever.invoke(query)
print(f'Query: {query}')
for i, doc in enumerate(resultados, 1):
    print(f'  {i}. [{doc.metadata["categoria"]}] {doc.page_content[:70]}...')

Query: ¿Cómo prevenir la diarrea en niños pequeños?
  1. [EDA] Cambio climático y proyección de brotes de EDA en la costa peruana. Mo...
  2. [Dengue] Dengue en gestantes: riesgo de complicaciones perinatales. El dengue d...
  3. [EDA] Saneamiento básico rural y su impacto en la incidencia de EDA. Distrit...
  4. [Dengue] Modelado matemático dengue con temperatura El Niño. Modelo SIR ampliad...
  5. [Malaria] Malaria gestacional: impacto en bajo peso al nacer. La malaria en gest...


## Precision@5 y latencia sobre el `eval_set` compartido

Mismo `eval_set` de 10 queries usado en Lab08 y Lab09.

In [ ]:
eval_set = [
    ('sales rehidratación oral rotavirus niños deshidratación', 'EDA'),
    ('resistencia medicamentos bacterias micobacterias segunda línea', 'Tuberculosis'),
    ('control criaderos mosquitos aedes fumigación vectorial', 'Dengue'),
    ('plasmodium tratamiento primaquina tafenoquina recaída', 'Malaria'),
    ('baciloscopia GeneXpert diagnóstico pulmonar esputo', 'Tuberculosis'),
    ('canal endémico brote epidémico semana epidemiológica', 'EDA'),
    ('serotipo circulante NS1 prueba diagnóstica laboratorio viral', 'Dengue'),
    ('gestante embarazo bajo peso nacer infección parasitaria', 'Malaria'),
    ('zinc lactancia materna prevención diarrea infantil nutrición', 'EDA'),
    ('alerta temprana predicción modelo satelital NDVI amazonia', 'Malaria'),
]

def precision_at_5(query, categoria_relevante):
    docs = vectorstore.similarity_search(query, k=5)
    categorias = [d.metadata['categoria'] for d in docs]
    return sum(1 for c in categorias if c == categoria_relevante) / 5

t0 = time.perf_counter()
precisions = [precision_at_5(q, cat) for q, cat in eval_set]
latencia_chroma_ms = (time.perf_counter() - t0) * 1000 / len(eval_set)
precision_chroma = float(np.mean(precisions))

print(f'Precision@5 = {precision_chroma:.4f}')
print(f'Latencia media = {latencia_chroma_ms:.2f} ms/query')

Precision@5 = 0.5600
Latencia media = 10.32 ms/query
